In [ ]:
#!/usr/bin/env python3
"""
v5 G0 — VCC leg. STEP 1: single streaming pass over adata_Training.h5ad.

RUN STEP 0 (kaggle_v5g0_vcc_00_probe_v1.py) FIRST and set VCC_X_MODE from its
`X_call` field. Getting that wrong costs a full 15 GB pass.

Environment (Kaggle, CPU only, free tier): numpy, pandas, h5py, scipy. No installs.
Peak RAM ~ 3-4 GB with the defaults below. Expect 30-90 min wall clock.

Input : /kaggle/input/<slug>/adata_Training.h5ad   (auto-discovered)
Output: /kaggle/working/v5g0_vcc_accum_v1.npz      (the only artifact STEP 2 needs)
        /kaggle/working/v5g0_vcc_extract_log_v1.json
        /kaggle/working/v5g0_vcc_batch_check_v1.csv

DESIGN (mirrors the scBloodNL leg so k is comparable across datasets):
  * cells of each perturbation are randomly split into disjoint halves A and B
  * "cells per perturbation" = cells per HALF; depth n therefore consumes 2n cells
  * non-targeting controls get their own independent A/B halves at the SAME depth
    (no shared control -> no leakage across the split)
  * depth slots are CUMULATIVE: summing slots 0..j gives the profile at DEPTHS[j];
    summing all slots gives that perturbation's full available depth
  * shuffled-label null: target labels permuted across ALL cells, preserving the
    cells-per-perturbation distribution exactly, then the identical pipeline
  * g is NOT computed here. It is recomputed inside each half in STEP 2.
"""
import json
import os
import sys
import time

import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp

# ---------------------------------------------------------------- config ----
H5AD = os.environ.get("VCC_H5AD", "")
if not H5AD:
    for root, _d, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".h5ad"):
                H5AD = os.path.join(root, f)
                break
        if H5AD:
            break
if not H5AD or not os.path.exists(H5AD):
    sys.exit("FATAL: no .h5ad found. Set env VCC_H5AD=...")

OUT = os.environ.get("VCC_OUT", "/kaggle/working")
os.makedirs(OUT, exist_ok=True)

# from STEP 0: "RAW_COUNTS" or "LOG_NORMALIZED"
X_MODE = os.environ.get("VCC_X_MODE", "RAW_COUNTS").upper()
assert X_MODE in ("RAW_COUNTS", "LOG_NORMALIZED"), X_MODE

# cells per HALF. Kept at/below the achievable max so the same perturbations
# contribute at every grid point (balanced panel); STEP 2 also reports the
# unbalanced panel. The tail slot holds everything above DEPTHS[-1].
DEPTHS = [int(x) for x in os.environ.get("VCC_DEPTHS", "25,50,100,200,400,600").split(",")]
REAL_SEEDS = [int(x) for x in os.environ.get("VCC_REAL_SEEDS", "0,1,2").split(",")]
NULL_SEEDS = [int(x) for x in os.environ.get("VCC_NULL_SEEDS", "100,101").split(",")]
CHUNK = int(os.environ.get("VCC_CHUNK", "5000"))       # cells per read block
# Held-out fraction used ONLY to rank perturbations by effect size. Keeping it
# disjoint from A and B is what stops the effect-size stratification from being
# circular with the reproducibility it is used to stratify.
C_FRAC = float(os.environ.get("VCC_C_FRAC", "0.2"))
TARGET_SUM = 1e4                                        # for the per-cell DEG normalisation

t0 = time.time()
log = {"h5ad": H5AD, "X_MODE": X_MODE, "DEPTHS": DEPTHS,
       "real_seeds": REAL_SEEDS, "null_seeds": NULL_SEEDS, "chunk": CHUNK}


def read_str_array(h5, key):
    node = h5[key]
    if isinstance(node, h5py.Group):
        cats = node["categories"][:]
        codes = node["codes"][:]
        cats = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in cats])
        return np.where(codes >= 0, cats[np.clip(codes, 0, None)], "<NA>")
    v = node[:]
    if v.dtype.kind in "SO":
        return np.array([x.decode() if isinstance(x, bytes) else str(x) for x in v])
    return v.astype(str)


# ------------------------------------------------------------ obs / var -----
with h5py.File(H5AD, "r") as h5:
    Xnode = h5["X"]
    if isinstance(Xnode, h5py.Group):
        enc = Xnode.attrs.get("encoding-type", b"")
        enc = enc.decode() if isinstance(enc, bytes) else str(enc)
        LAYOUT = "csr" if "csr" in enc else ("csc" if "csc" in enc else "unknown")
        SHAPE = tuple(int(v) for v in Xnode.attrs["shape"])
    else:
        LAYOUT = "dense"
        SHAPE = tuple(int(v) for v in Xnode.shape)
    if LAYOUT not in ("dense", "csr"):
        sys.exit(f"FATAL: X layout '{LAYOUT}' is not row-streamable by this script. "
                 "Re-save the matrix as CSR (or dense) and rerun. Do NOT guess a "
                 "column-major path here — an untested fallback is worse than none.")

    target = read_str_array(h5, "obs/target_gene")
    obs_cols = list(h5["obs"].attrs.get("column-order", []))
    obs_cols = [c.decode() if isinstance(c, bytes) else str(c) for c in obs_cols]
    batch = read_str_array(h5, "obs/batch") if "batch" in obs_cols else None
    vidx = h5["var"].attrs.get("_index", b"_index")
    vidx = vidx.decode() if isinstance(vidx, bytes) else str(vidx)
    genes = read_str_array(h5, f"var/{vidx}")

N_CELLS, N_GENES = SHAPE
assert len(target) == N_CELLS, (len(target), N_CELLS)
LAYOUT_LOG = {"layout": LAYOUT, "shape": [N_CELLS, N_GENES]}
log["matrix"] = LAYOUT_LOG

# ---- group coding: control first (index 0), then perturbations -------------
CTRL_LABEL = None
for cand in ("non-targeting", "non_targeting", "nontargeting", "NTC", "control"):
    if (target == cand).any():
        CTRL_LABEL = cand
        break
if CTRL_LABEL is None:
    sys.exit(f"FATAL: no control label found in target_gene. Levels: {sorted(set(target))[:20]}")

pert_labels = sorted(set(target) - {CTRL_LABEL})
groups = [CTRL_LABEL] + pert_labels
gcode = {g: i for i, g in enumerate(groups)}
group_idx = np.array([gcode[t] for t in target], dtype=np.int32)
N_GROUP = len(groups)
CTRL = 0
PERT_IDX = list(range(1, N_GROUP))

N_HALF = 3          # partition 0 = A, 1 = B, 2 = C (held-out effect-size panel)
N_SLOT = len(DEPTHS) + 1
N_BIN = N_GROUP * N_HALF * N_SLOT

counts_per_group = np.bincount(group_idx, minlength=N_GROUP)
log["design"] = {
    "control_label": CTRL_LABEL, "n_perturbations": len(pert_labels),
    "n_control_cells": int(counts_per_group[CTRL]),
    "n_perturbed_cells": int(counts_per_group[1:].sum()),
    "cells_per_pert_median": float(np.median(counts_per_group[1:])),
    "cells_per_pert_min": int(counts_per_group[1:].min()),
    "cells_per_pert_max": int(counts_per_group[1:].max()),
    "n_bins": N_BIN, "n_slots": N_SLOT,
}
print(json.dumps(log["design"], indent=2))


def bin_id(g, half, slot):
    return (g * N_HALF + half) * N_SLOT + slot


def assign(group_codes, seed):
    """-> (bin_of_cell int32 [-1 = unused], avail[g,part] int32)

    Within each group the cells are permuted; a fraction C_FRAC is set aside as
    the held-out effect-size panel C, and the remainder is split into two
    disjoint halves A and B of exactly equal size (an odd cell is dropped).
    Each cell's rank inside its half maps to a cumulative depth slot; all of C
    sits in slot 0 and is always used at full depth."""
    rng = np.random.default_rng(seed)
    b = np.full(group_codes.size, -1, dtype=np.int32)
    avail = np.zeros((N_GROUP, N_HALF), dtype=np.int32)
    order = np.argsort(group_codes, kind="stable")
    bounds = np.searchsorted(group_codes[order], np.arange(N_GROUP + 1))
    dep = np.asarray(DEPTHS)
    for g in range(N_GROUP):
        idx = rng.permutation(order[bounds[g]:bounds[g + 1]])
        nc = int(round(C_FRAC * idx.size))
        cpanel, rest = idx[:nc], idx[nc:]
        if cpanel.size:
            b[cpanel] = (g * N_HALF + 2) * N_SLOT + 0
            avail[g, 2] = cpanel.size
        nh = rest.size // 2
        if nh == 0:
            continue
        slot = np.searchsorted(dep, np.arange(nh), side="right")
        for half, sel in ((0, rest[:nh]), (1, rest[nh:2 * nh])):
            b[sel] = (g * N_HALF + half) * N_SLOT + slot
            avail[g, half] = nh
    return b, avail


# ---- build every run's assignment up front (obs-only, cheap) ---------------
runs = []
for s in REAL_SEEDS:
    bmap, avail = assign(group_idx, s)
    runs.append({"kind": "real", "seed": s, "bin": bmap, "avail": avail})
for s in NULL_SEEDS:
    rng = np.random.default_rng(10_000 + s)
    shuffled = group_idx[rng.permutation(group_idx.size)]   # preserves group sizes
    bmap, avail = assign(shuffled, s)
    runs.append({"kind": "shuffled", "seed": s, "bin": bmap, "avail": avail})
N_RUN = len(runs)
print(f"{N_RUN} runs assigned  ({len(REAL_SEEDS)} real, {len(NULL_SEEDS)} shuffled)")

# ---- batch check: is the A/B split independent of batch? -------------------
batch_rows = []
if batch is not None:
    from scipy.stats import chi2_contingency
    for r in runs[: len(REAL_SEEDS)]:
        half_of_cell = np.where(r["bin"] >= 0, (r["bin"] // N_SLOT) % N_HALF, -1)
        half_of_cell = np.where(half_of_cell == 2, -1, half_of_cell)   # C is not part of the A/B test
        for g in range(N_GROUP):
            m = (group_idx == g) & (half_of_cell >= 0)
            if m.sum() < 20:
                continue
            ct = pd.crosstab(batch[m], half_of_cell[m])
            if ct.shape[0] < 2 or ct.shape[1] < 2:
                p = np.nan
            else:
                p = float(chi2_contingency(ct.values)[1])
            batch_rows.append(dict(seed=r["seed"], group=groups[g], n_batches=int(ct.shape[0]),
                                   chi2_p_half_vs_batch=p))
    pd.DataFrame(batch_rows).to_csv(os.path.join(OUT, "v5g0_vcc_batch_check_v1.csv"), index=False)
    bp = pd.DataFrame(batch_rows)["chi2_p_half_vs_batch"].dropna()
    log["batch_check"] = {
        "n_tests": int(bp.size),
        "frac_p_below_0.05": float((bp < 0.05).mean()) if bp.size else None,
        "expected_under_null": 0.05,
        "note": "split is random within perturbation, so this should sit at ~5%.",
    }
    print("batch check:", log["batch_check"])

# ------------------------------------------------------------ accumulate ----
accum = np.zeros((N_RUN, N_BIN, N_GENES), dtype=np.float64)     # summed profile
bin_n = np.zeros((N_RUN, N_BIN), dtype=np.int64)                # cells per bin
detect = np.zeros(N_GENES, dtype=np.int64)                      # cells with x>0
lib_sum = np.zeros(N_CELLS, dtype=np.float64)
# per-cell log-normalised moments per group, restricted to the HELD-OUT C panel
# of the first real run -> Welch t-test for the DEG bins, independent of A/B.
C_MASK = ((runs[0]["bin"] >= 0) & (((runs[0]["bin"] // N_SLOT) % N_HALF) == 2))
print(f"held-out effect-size panel C: {int(C_MASK.sum())} cells ({C_FRAC:.0%})")
deg_n = np.zeros(N_GROUP, dtype=np.int64)
deg_sum = np.zeros((N_GROUP, N_GENES), dtype=np.float64)
deg_sumsq = np.zeros((N_GROUP, N_GENES), dtype=np.float64)

print(f"accumulators: {accum.nbytes / 1e9:.2f} GB")

with h5py.File(H5AD, "r") as h5:
    X = h5["X"]
    if LAYOUT == "csr":
        indptr = X["indptr"][:]
    for start in range(0, N_CELLS, CHUNK):
        stop = min(start + CHUNK, N_CELLS)
        if LAYOUT == "dense":
            block = np.asarray(X[start:stop, :], dtype=np.float32)
        else:
            lo, hi = int(indptr[start]), int(indptr[stop])
            block = sp.csr_matrix(
                (X["data"][lo:hi], X["indices"][lo:hi], indptr[start:stop + 1] - lo),
                shape=(stop - start, N_GENES)).toarray().astype(np.float32)

        detect += (block > 0).sum(0).astype(np.int64)
        lib = block.sum(1, dtype=np.float64)
        lib_sum[start:stop] = lib

        # per-cell normalised values for the DEG test
        if X_MODE == "RAW_COUNTS":
            with np.errstate(divide="ignore", invalid="ignore"):
                norm = np.log1p(block * (TARGET_SUM / np.where(lib > 0, lib, 1))[:, None])
        else:
            norm = block
        cm = C_MASK[start:stop]
        if cm.any():
            gcode_chunk = group_idx[start:stop][cm]
            cols = np.nonzero(cm)[0]
            Mg = sp.csr_matrix(
                (np.ones(cols.size, dtype=np.float64), (gcode_chunk, np.arange(cols.size))),
                shape=(N_GROUP, cols.size))
            nc = norm[cols].astype(np.float64)
            deg_sum += Mg @ nc
            deg_sumsq += Mg @ (nc ** 2)
            deg_n += np.bincount(gcode_chunk, minlength=N_GROUP)
            del nc
        del norm

        # the profile accumulators, one indicator matmul per run
        payload = block.astype(np.float64)      # raw counts, or log values if LOG_NORMALIZED
        for ri, r in enumerate(runs):
            bc = r["bin"][start:stop]
            keep = bc >= 0
            if not keep.any():
                continue
            rows = bc[keep]
            cols = np.nonzero(keep)[0]
            M = sp.csr_matrix((np.ones(rows.size, dtype=np.float64), (rows, cols)),
                              shape=(N_BIN, stop - start))
            accum[ri] += M @ payload
            bin_n[ri] += np.bincount(rows, minlength=N_BIN)
        del block, payload

        if (start // CHUNK) % 5 == 0:
            el = time.time() - t0
            print(f"  {stop}/{N_CELLS} cells  {el:6.1f}s  eta {el * (N_CELLS - stop) / max(stop, 1):6.1f}s",
                  flush=True)

log["elapsed_stream_s"] = round(time.time() - t0, 1)
log["library_size"] = {"median": float(np.median(lib_sum)), "mean": float(lib_sum.mean()),
                       "min": float(lib_sum.min()), "max": float(lib_sum.max())}

np.savez_compressed(
    os.path.join(OUT, "v5g0_vcc_accum_v1.npz"),
    accum=accum.astype(np.float32), bin_n=bin_n, detect=detect,
    genes=np.array(genes), groups=np.array(groups), depths=np.array(DEPTHS),
    n_slot=N_SLOT, n_half=N_HALF, n_group=N_GROUP,
    run_kind=np.array([r["kind"] for r in runs]),
    run_seed=np.array([r["seed"] for r in runs]),
    avail=np.stack([r["avail"] for r in runs]),
    counts_per_group=counts_per_group, n_cells=N_CELLS,
    deg_n=deg_n, deg_sum=deg_sum, deg_sumsq=deg_sumsq,
    x_mode=X_MODE, ctrl_label=CTRL_LABEL, c_frac=C_FRAC,
    deg_panel="heldout_C_of_first_real_run",
    umi_median=float(np.median(lib_sum)),
)
with open(os.path.join(OUT, "v5g0_vcc_extract_log_v1.json"), "w") as f:
    json.dump(log, f, indent=2, default=str)

print("\nWROTE", os.path.join(OUT, "v5g0_vcc_accum_v1.npz"))
print(json.dumps(log, indent=2, default=str))
print(f"total {time.time() - t0:.1f}s")